# Step 4 · Prediction: can we know in advance which counties' activity will collapse?

Steps 2 and 3 answered "what happened on average". This step asks a different question: **in a new storm, using only data available while the storm is happening, can we predict which county-weeks will see a large drop in visits?** This is what emergency managers need: outage data (EAGLE-I) and wind and rain data are close to real time, and the pre-storm mobility network is already known.

| Question | Approach |
| --- | --- |
| **P1 Can it predict across storms?** | Gradient-boosted trees (`HistGradientBoostingRegressor`). **Leave one storm out**: train on 6 storms, predict the unseen 7th, and rotate through all 7 |
| **P2 Does the mobility network help?** | Two models compared: "local" uses only the county's own outage, wind and rain and county attributes; "network" adds features from the pre-storm mobility network (the share of usual visitors who come from counties without power, whether the places residents usually visit are without power, and so on) |
| **P3 Does it transfer to Uri?** | Train on all 7 storms and predict the February 2021 Texas winter storm Uri directly. Uri was a grid collapse without wind damage, a situation the training data never contains |
| **P4 Whose errors?** | Among county-weeks with 24+ hours of outage, compare prediction errors for rural counties (under 50,000 people) and the rest, to see whether the model systematically underestimates rural losses |

**What is predicted**: each county-week's change in visits (`visits_in_chg`) minus the median for counties in the same event and week that had no substantial outage over the whole event, i.e. "how much more it fell than normal counties at the same time". Statewide shocks (all of Texas staying home during Uri, say) are thus not counted, which is the same idea as Step 2's event × week fixed effects.

**Benchmark**: a linear model with outage-duration bands only (equivalent to Step 2's outage-only specification). A machine-learning model is useful only if it beats this.

**Structure**: A data and features → B the decision rules, fixed in advance → C leave one storm out → D transfer to Uri → E what the model learned → F whose errors → G checklist and conclusions. No Earth Engine is needed; it runs in a few minutes.

In [ ]:
# 1. Environment, folders, chart style
import os, sys, glob, zipfile, subprocess
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.metrics import average_precision_score
try:
    import geopandas as gpd
except ImportError:                     # Colab usually has geopandas; install it if not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True)
    import geopandas as gpd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."
OUT = f"{BASE}/outage_mobility"
for f in ["step1b_events.csv", "step1b_outage_event.csv", "step1b_panel_weekly.csv.gz", "step2a_hazard_weekly.csv",
          "geods_weeks/week_totals.csv", "oi_geoids_county.csv"]:
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run the earlier steps first.")

def tidy(df, n=2):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_DARK, C_MID, C_LIGHT, C_GREY = "#104281", "#2a78d6", "#86b6ef", "#7a7974"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
TRANSFER = "uri2021"                           # used only to test transfer, never for training
WEEK_FROM, WEEK_TO = -5, 6
BASE_COLS = [-5, -4, -3, -2]
TRAIN_FROM = -1                                # training uses weeks −1 to 6 (baseline weeks are near 0 by construction and are left out)
EVAL_WEEKS = [0, 1, 2]                         # evaluation looks only at the first three weeks after the storm
Y_CLIP = 150                                   # the outcome is clipped at ±150 log points to limit extreme noise
BIG_DROP = -20                                 # a "large drop": 20+ log points (about 18%) below normal counties in the same week
RURAL_POP = 50_000                             # counties below this population count as rural
AFFECTED_HOURS = 24                            # P4 looks only at county-weeks with 24+ hours of outage
HIT_HOURS = 72
B_BOOT = 999
SEED = 2026
GBM = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)
# hyperparameters are fixed in advance with no tuning: tuning would use the held-out storm, which would then no longer be unseen
OUT_EDGES  = [0, 6, 24, 72, 120, np.inf]
OUT_LEVELS = ["h000", "h006", "h024", "h072", "h120"]
CUM_FROM = -1

## A. Data and features

As before, counties with baseline problems are left out (`flag_base`, `flag_prior`, `low_sample`), but this time **Uri is kept**, only not used for training.

Features come in three groups, all available while a storm is under way:

| Group | Features |
| --- | --- |
| Outage | Hours without power this week, last week, and cumulatively since week −1 |
| Wind, rain and county attributes | Gust and rainfall this week and cumulatively, frozen precipitation, lowest temperature, weeks since the storm; population, population density, distance to the coast, usual visits per person |
| **Network** (used only by the "network" model) | Share of usual visitors from other counties; share of residents' usual visits made in other counties; **inbound exposure**: the average outage hours this week of the counties usual visitors come from (weighted by visitors), and the share of them with 72+ h; **outbound exposure**: the average outage hours this week of the counties residents usually visit, and the share of them with 72+ h |

All network weights come from baseline county-pair flows (weeks −5 to −2). Counties outside the study area have no outage data and are treated as having power.

In [ ]:
# 3. Sample, outcome, outage and hazard features
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
EV_ALL = list(EVENTS.index)
EV_TRAIN = [k for k in EV_ALL if k != TRANSFER]
W  = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", dtype={"fips": str})
L  = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})
HZ = pd.read_csv(f"{OUT}/step2a_hazard_weekly.csv", dtype={"fips": str})

HZ = HZ.sort_values(["event", "fips", "rel_week"])
after = HZ["rel_week"] >= CUM_FROM
grp = [HZ["event"], HZ["fips"]]
HZ["gust_cum"] = HZ["gust_mean_ms"].where(after).groupby(grp).cummax().fillna(HZ["gust_mean_ms"])
HZ["rain_cum"] = HZ["rain_mm"].where(after, 0).groupby(grp).cumsum().where(after, HZ["rain_mm"])
D = (W.merge(L[["event", "fips", "name", "group", "flag_base", "flag_prior"]], on=["event", "fips"])
      .merge(HZ[["event", "fips", "rel_week", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "tmin_c", "gust_cum", "rain_cum"]],
             on=["event", "fips", "rel_week"], how="left")
      .sort_values(["event", "fips", "rel_week"]).reset_index(drop=True))
g = D.groupby(["event", "fips"])["out_week_hours"]
D["out_prev_hours"] = g.shift(1)
D["out_cum_hours"] = D["out_week_hours"].where(D["rel_week"] >= CUM_FROM, 0).groupby([D["event"], D["fips"]]).cumsum()
D["out_bin"] = pd.cut(D["out_week_hours"], OUT_EDGES, right=False, labels=OUT_LEVELS).astype(str)
D["keep"] = ~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D["out_week_hours"].notna() & D["visits_in_chg"].notna()
REF = D[D["keep"] & (D["group"] == "none")].groupby(["event", "rel_week"])["visits_in_chg"].median().rename("ref")
D = D.join(REF, on=["event", "rel_week"])
D["y"] = (D["visits_in_chg"] - D["ref"]).clip(-Y_CLIP, Y_CLIP)       # how much more it fell than normal counties that week
print(f"Reference counties (no substantial outage over the event) per event-week, minimum: "
      f"{int(D[D['keep'] & (D['group'] == 'none')].groupby(['event', 'rel_week']).size().min())}")

In [ ]:
# 4. County attributes: population, area, distance to the coast
GEO = f"{OUT}/step2c_geo"
COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"

def county_table():
    # use the county boundaries downloaded in Step 2c (only the land area in the attribute table); download them if missing
    path = f"{GEO}/county.zip"
    if not os.path.exists(path) or os.path.getsize(path) < 10_000:
        os.makedirs(GEO, exist_ok=True)
        r = requests.get(COUNTY_URL, timeout=600, headers={"User-Agent": "Mozilla/5.0 (research script)"})
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    folder = f"{GEO}/county"
    with zipfile.ZipFile(path) as z:
        z.extractall(folder)
    t = gpd.read_file(glob.glob(f"{folder}/*.shp")[0], ignore_geometry=True)
    return pd.Series(t["ALAND"].astype(float).to_numpy() / 1e6, index=t["GEOID"].astype(str), name="area_km2")

AREA = county_table()
OI = pd.read_csv(f"{OUT}/oi_geoids_county.csv")
OI["fips"] = OI["countyfips"].astype(int).astype(str).str.zfill(5)
POP = OI.set_index("fips")["county_pop2019"].astype(float)
COAST_FILE = f"{OUT}/step2c_coast_distance.csv"
COAST = pd.read_csv(COAST_FILE, dtype={"fips": str}).set_index("fips")["coast_km"] if os.path.exists(COAST_FILE) else pd.Series(dtype=float)
D["pop"] = D["fips"].map(POP)
D["log_pop"] = np.log(D["pop"])
D["log_density"] = np.log(D["pop"] / D["fips"].map(AREA))
D["coast_km"] = D["fips"].map(COAST)
D["rural"] = D["pop"] < RURAL_POP
print(f"County-weeks with population: {D['pop'].notna().mean():.1%}; with area: {D['fips'].map(AREA).notna().mean():.1%}; with distance to the coast: {D['coast_km'].notna().mean():.1%}")

In [ ]:
# 5. Network features (county-pair flows of each event)
TOT = pd.read_csv(f"{OUT}/geods_weeks/week_totals.csv", parse_dates=["week"]).set_index("week")["total_visits"].astype(float)
hours_all = W[["event", "fips", "rel_week", "out_week_hours"]].fillna({"out_week_hours": 0.0})
NET, STATIC = [], []
for key in EV_ALL:
    area = set(D.loc[D["event"] == key, "fips"])
    E = pd.read_csv(f"{OUT}/step1b_edges_{key}.csv.gz", dtype={"geoid_o": str, "geoid_d": str}, parse_dates=["week"])
    E = E[E["rel_week"].isin(BASE_COLS)]
    E["f"] = E["visitor_flows"] / E["week"].map(TOT).to_numpy() * 1e6
    B = (E.groupby(["geoid_o", "geoid_d"])["f"].sum() / len(BASE_COLS)).rename("f").reset_index()   # mean weekly flow over the baseline
    home = B["geoid_o"] == B["geoid_d"]
    res_tot = B[B["geoid_o"].isin(area)].groupby("geoid_o")["f"].sum()
    out_tot = B[~home & B["geoid_o"].isin(area)].groupby("geoid_o")["f"].sum()
    in_tot = B[B["geoid_d"].isin(area)].groupby("geoid_d")["f"].sum()
    in_non = B[~home & B["geoid_d"].isin(area)].groupby("geoid_d")["f"].sum()
    STATIC.append(pd.DataFrame({"event": key, "fips": sorted(area)}).assign(
        base_out_share=lambda d: d["fips"].map(out_tot / res_tot),
        base_visitor_share=lambda d: d["fips"].map(in_non / in_tot),
        log_base_visits_pc=lambda d: np.log(d["fips"].map(in_tot) / d["fips"].map(POP))))
    hrs = hours_all[hours_all["event"] == key].drop(columns="event")
    po = B[~home & B["geoid_o"].isin(area)].copy()
    po["w"] = po["f"] / po["geoid_o"].map(out_tot)                               # residents' usual split across other counties
    po = po.merge(hrs.rename(columns={"fips": "geoid_d"}), on="geoid_d")          # destinations outside the study area have no outage data and drop out of the sums (= treated as having power)
    po["w72"] = po["w"] * (po["out_week_hours"] >= HIT_HOURS)
    po["wh"] = po["w"] * po["out_week_hours"]
    outb = po.groupby(["geoid_o", "rel_week"])[["wh", "w72"]].sum().rename(columns={"wh": "net_out_hours", "w72": "net_out_72"})
    pi = B[~home & B["geoid_d"].isin(area)].copy()
    pi["w"] = pi["f"] / pi["geoid_d"].map(in_tot)                                 # usual share of visitors from each county
    pi = pi.merge(hrs.rename(columns={"fips": "geoid_o"}), on="geoid_o")
    pi["w72"] = pi["w"] * (pi["out_week_hours"] >= HIT_HOURS)
    pi["wh"] = pi["w"] * pi["out_week_hours"]
    inb = pi.groupby(["geoid_d", "rel_week"])[["wh", "w72"]].sum().rename(columns={"wh": "net_in_hours", "w72": "net_in_72"})
    both = outb.rename_axis(["fips", "rel_week"]).join(inb.rename_axis(["fips", "rel_week"]), how="outer").reset_index().assign(event=key)
    NET.append(both)
    print(f"{key}: {len(B):,} baseline county pairs")
NET = pd.concat(NET, ignore_index=True)
STATIC = pd.concat(STATIC, ignore_index=True)
D = D.drop(columns=[c for c in NET.columns if c.startswith("net_")] + ["base_out_share", "base_visitor_share", "log_base_visits_pc"], errors="ignore")
D = D.merge(NET, on=["event", "fips", "rel_week"], how="left").merge(STATIC, on=["event", "fips"], how="left")
for c in ["net_out_hours", "net_out_72", "net_in_hours", "net_in_72"]:
    D[c] = D[c].fillna(0.0)

In [ ]:
# 6. Feature table
F_LOCAL = ["out_week_hours", "out_prev_hours", "out_cum_hours", "gust_mean_ms", "gust_cum", "rain_mm", "rain_cum", "cold_ppt_mm", "tmin_c",
           "rel_week", "log_pop", "log_density", "coast_km", "log_base_visits_pc"]
F_NET = F_LOCAL + ["base_visitor_share", "base_out_share", "net_in_hours", "net_in_72", "net_out_hours", "net_out_72"]
S = D[D["keep"] & D["rel_week"].between(TRAIN_FROM, WEEK_TO) & D["y"].notna()].reset_index(drop=True)
S["is_eval"] = S["rel_week"].isin(EVAL_WEEKS)
S["big"] = S["y"] <= BIG_DROP
summary = S[S["is_eval"]].groupby("event").agg(county_weeks=("y", "size"), big_drops=("big", "sum"), hit_72=("out_week_hours", lambda h: int((h >= HIT_HOURS).sum())))
print("Features with the most missing values (gradient-boosted trees handle missing values directly):",
      S[F_NET].isna().mean().sort_values(ascending=False).head(4).round(3).to_dict())
summary

## B. The decision rules, fixed in advance

Written down before looking at any result. The "evaluation set" is the county-weeks of the first three weeks after the storm (weeks 0–2). A "large drop" is a fall of 20+ log points more than normal counties in the same week.

- **P1 Predicts across storms**: pooled over the leave-one-storm-out folds, the "network" model has R² > 0 and an average precision for "large drops" at least 3 times the chance level (the share of large drops)
- **P2 The network helps**: in finding "large drops", the "network" model's average precision is higher than the "local" model's, with a 95% interval from resampling counties that excludes 0, and higher in a majority of the storms with at least 3 large drops. (Average precision is used rather than mean absolute error because most county-weeks had no outage at all, so their errors are mostly noise and dilute any difference; finding the places that collapse is also what emergency managers need)
- **P3 Transfers to Uri**: the "network" model trained on the 7 storms predicts Uri with R² > 0 and an average precision at least 3 times the chance level
- **P4 Whose errors**: among county-weeks with 24+ hours of outage, the mean residual (actual − predicted) of rural counties minus that of other counties has a 95% interval excluding 0. Below 0 means the model **underestimates rural losses**

How much the "network" model beats the linear benchmark is reported alongside, for reference.

## C. Leave one storm out

Each time, one storm is removed, the three models (linear benchmark, "local", "network") are trained on the other 6, and the removed storm is predicted. The table shows each model's performance on the evaluation set of each storm:

- **R²**: how much better than predicting the mean everywhere (can be below 0)
- **MAE**: mean absolute error (log points)
- **Spearman**: rank correlation between predicted and actual
- **AP**: average precision in ranking "large drop" county-weeks first, against the chance level (`base`)

In [ ]:
# 7. Three models; leave one storm out
def linear_bins(train, test):
    # benchmark: a linear model with outage-duration bands only
    X = lambda d: pd.get_dummies(d["out_bin"], dtype=float).reindex(columns=OUT_LEVELS, fill_value=0.0).to_numpy()
    b = np.linalg.lstsq(X(train), train["y"].to_numpy(), rcond=None)[0]
    return X(test) @ b

def gbm_fit(train, feats):
    return HistGradientBoostingRegressor(**GBM).fit(train[feats], train["y"])

def metrics(d, col):
    y, p = d["y"].to_numpy(), d[col].to_numpy()
    big = y <= BIG_DROP
    return {"r2": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(), "mae": np.abs(y - p).mean(),
            "spearman": pd.Series(y).rank().corr(pd.Series(p).rank()),
            "ap": average_precision_score(big, -p) if big.any() else np.nan, "base": big.mean(), "n": len(y), "n_big": int(big.sum())}

MODELS = {"p_linear": "Linear (outage bands)", "p_local": "GBM, local", "p_net": "GBM, + network"}
parts, FOLD_MODELS = [], {}
for key in EV_TRAIN:
    tr, te = S[S["event"].isin(EV_TRAIN) & (S["event"] != key)], S[S["event"] == key].copy()
    te["p_linear"] = linear_bins(tr, te)
    te["p_local"] = gbm_fit(tr, F_LOCAL).predict(te[F_LOCAL])
    FOLD_MODELS[key] = gbm_fit(tr, F_NET)
    te["p_net"] = FOLD_MODELS[key].predict(te[F_NET])
    parts.append(te)
LOSO = pd.concat(parts, ignore_index=True)
EV_SET = LOSO[LOSO["is_eval"]]
rows = [{"event": key, "model": col, **metrics(EV_SET[EV_SET["event"] == key], col)} for key in EV_TRAIN for col in MODELS]
rows += [{"event": "pooled", "model": col, **metrics(EV_SET, col)} for col in MODELS]
PERF = pd.DataFrame(rows)
LOSO[["event", "fips", "rel_week", "y", "out_week_hours", "rural", "is_eval"] + list(MODELS)].to_csv(f"{OUT}/step4_predictions_loso.csv", index=False)
tidy(PERF.pivot_table(index="event", columns="model", values=["r2", "spearman", "ap"]).reindex(EV_TRAIN + ["pooled"]), 2)

In [ ]:
# 8. P1 and P2
pooled = PERF[PERF["event"] == "pooled"].set_index("model")
P1 = bool(pooled.loc["p_net", "r2"] > 0 and pooled.loc["p_net", "ap"] >= 3 * pooled.loc["p_net", "base"])
big = (EV_SET["y"] <= BIG_DROP).to_numpy()
cid = pd.factorize(EV_SET["fips"])[0]
rng = np.random.default_rng(SEED)
boot = []
for _ in range(B_BOOT):                                                    # resample counties: a county drawn k times gets weight k on its rows
    wgt = np.bincount(rng.integers(0, cid.max() + 1, cid.max() + 1), minlength=cid.max() + 1)[cid]
    if big[wgt > 0].any():
        boot.append(average_precision_score(big, -EV_SET["p_net"], sample_weight=wgt) - average_precision_score(big, -EV_SET["p_local"], sample_weight=wgt))
DIFF = pooled.loc["p_net", "ap"] - pooled.loc["p_local", "ap"]
DIFF_LO, DIFF_HI = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
per = PERF[(PERF["event"] != "pooled") & (PERF["n_big"] >= 3)].pivot(index="event", columns="model", values="ap")
WINS, ELIG = int((per["p_net"] > per["p_local"]).sum()), len(per)       # only storms with at least 3 large drops count
P2 = bool(DIFF_LO > 0 and WINS > ELIG / 2)
MAE_DIFF = pooled.loc["p_net", "mae"] - pooled.loc["p_local", "mae"]
print(f"P1 predicts across storms: {P1} (R² {pooled.loc['p_net', 'r2']:.3f}; AP {pooled.loc['p_net', 'ap']:.3f}, chance {pooled.loc['p_net', 'base']:.3f})")
print(f"P2 the network helps: {P2} (AP difference {DIFF:+.3f} [{DIFF_LO:+.3f}, {DIFF_HI:+.3f}]; better in {WINS} of the {ELIG} storms with enough large drops; MAE difference {MAE_DIFF:+.3f})")
print(f"For reference: \"network\" model vs linear benchmark, pooled R² {pooled.loc['p_net', 'r2']:.3f} vs {pooled.loc['p_linear', 'r2']:.3f}, AP {pooled.loc['p_net', 'ap']:.3f} vs {pooled.loc['p_linear', 'ap']:.3f}")

## D. Transfer to Uri

All 7 storms are used for training and Uri's weeks 0–2 are predicted. Uri had no wind damage; the outages came from ERCOT's rotating outages and frozen power plants, with snow and ice across all of Texas. The training data hold almost no freezing weather (only the Oklahoma ice storm), so this is a strict test of transfer.

In [ ]:
# 9. P3: transfer to Uri
tr_all = S[S["event"].isin(EV_TRAIN)]
URI = S[S["event"] == TRANSFER].copy()
URI["p_linear"] = linear_bins(tr_all, URI)
URI["p_local"] = gbm_fit(tr_all, F_LOCAL).predict(URI[F_LOCAL])
MODEL_ALL = gbm_fit(tr_all, F_NET)
URI["p_net"] = MODEL_ALL.predict(URI[F_NET])
URI[["event", "fips", "rel_week", "y", "out_week_hours", "rural", "is_eval"] + list(MODELS)].to_csv(f"{OUT}/step4_predictions_uri.csv", index=False)
UE = URI[URI["is_eval"]]
uri_perf = pd.DataFrame([{"event": TRANSFER, "model": col, **metrics(UE, col)} for col in MODELS])
PERF = pd.concat([PERF[PERF["event"] != TRANSFER], uri_perf], ignore_index=True)
PERF.to_csv(f"{OUT}/step4_performance.csv", index=False)
u = uri_perf.set_index("model").loc["p_net"]
P3 = bool(u["r2"] > 0 and u["ap"] >= 3 * u["base"]) if u["n_big"] > 0 else None
print(f"P3 transfers to Uri: {P3} (R² {u['r2']:.3f}, Spearman {u['spearman']:.2f}, AP {u['ap']:.3f}, chance {u['base']:.3f}, {int(u['n_big'])} large drops)")
tidy(uri_perf.set_index("model")[["r2", "mae", "spearman", "ap", "base", "n", "n_big"]], 3)

In [ ]:
# 10. Figure 1: performance by storm; Figure 2: predicted vs actual
order = EV_TRAIN + ["pooled", TRANSFER]
names = {k: EVENTS.loc[k, "name"] for k in EV_ALL} | {"pooled": "Pooled, 7 held-out storms"}
STY = {"p_linear": (C_GREY, "s", "Linear, outage bands only"), "p_local": (C_LIGHT, "o", "Gradient boosting, local features"),
       "p_net": (C_DARK, "o", "Gradient boosting, + network features")}
fig, axes = plt.subplots(1, 2, figsize=(13, 0.5 * len(order) + 1.6), sharey=True)
ys = np.arange(len(order))[::-1]
for ax, metric, xlabel in [(axes[0], "r2", "Out-of-sample R²"), (axes[1], "ap", "Average precision for large drops (≤ −20 log points)")]:
    for col, (color, mk, label) in STY.items():
        v = PERF[PERF["model"] == col].set_index("event").reindex(order)[metric]
        off = {"p_linear": 0.2, "p_local": 0.0, "p_net": -0.2}[col]
        ax.scatter(v, ys + off, color=color, marker=mk, s=36, zorder=3, label=label)
    if metric == "ap":
        base = PERF[PERF["model"] == "p_net"].set_index("event").reindex(order)["base"]
        ax.scatter(base, ys, marker="|", s=120, color=INK, zorder=2, label="Chance level")
    ax.axhline(1.5, color=FAINT, lw=1)
    ax.axhline(0.5, color=FAINT, lw=1, ls=":")
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(ys)
axes[0].set_yticklabels([names[k] + (" (transfer)" if k == TRANSFER else "") for k in order])
handles, labs = axes[1].get_legend_handles_labels()
fig.legend(handles, labs, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0.0))
axes[0].set_title("Held-out storms (each predicted by a model trained on the other 6)")
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(f"{OUT}/step4_fig1_performance.png"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, d, title in [(axes[0], EV_SET, "7 storms, each held out"), (axes[1], UE, "Uri, trained on the 7 storms")]:
    hit = d["out_week_hours"] >= HIT_HOURS
    ax.scatter(d.loc[~hit, "p_net"], d.loc[~hit, "y"], s=6, color=C_LIGHT, alpha=0.35, edgecolor="none", label="Under 72 h of outage")
    ax.scatter(d.loc[hit, "p_net"], d.loc[hit, "y"], s=18, color=C_DARK, edgecolor="none", label="72+ h of outage")
    lim = [-Y_CLIP, 60]
    ax.plot(lim, lim, color=FAINT, lw=1)
    ax.axhline(BIG_DROP, color=FAINT, lw=1, ls="--")
    ax.set_xlim(-100, 40)
    ax.set_ylim(-Y_CLIP - 5, 60)
    ax.set_title(title)
    ax.set_xlabel("Predicted change vs normal counties (log points)")
axes[0].set_ylabel("Actual change vs normal counties (log points)")
axes[0].legend(fontsize=8.5, loc="upper left")
fig.tight_layout()
fig.savefig(f"{OUT}/step4_fig2_predicted_actual.png"); plt.show()

## E. What the model learned

- **Permutation importance**: on the held-out storm of each fold, one feature's values are shuffled and the increase in mean absolute error is recorded; averaged over the 7 folds. The larger the increase, the more the model relies on that feature
- **Partial dependence**: for the "network" model trained on all 7 storms, how the prediction changes with outage hours and with inbound exposure (the share of usual visitors from counties with 72+ h of outage), other features kept at their actual values

In [ ]:
# 11. Permutation importance and partial dependence; Figure 3
imp = []
for key, m in FOLD_MODELS.items():
    te = EV_SET[EV_SET["event"] == key]
    r = permutation_importance(m, te[F_NET], te["y"], scoring="neg_mean_absolute_error", n_repeats=5, random_state=SEED)
    imp.append(pd.Series(r.importances_mean, index=F_NET, name=key))
IMP = pd.concat(imp, axis=1)
IMP["mean"] = IMP.mean(axis=1)
IMP = IMP.sort_values("mean", ascending=False)
IMP.to_csv(f"{OUT}/step4_importance.csv")
X_all = tr_all[F_NET]
pd_h = partial_dependence(MODEL_ALL, X_all, ["out_week_hours"], grid_resolution=40, percentiles=(0, 1))     # from the minimum to the maximum
pd_e = partial_dependence(MODEL_ALL, X_all, ["net_in_72"], grid_resolution=30, percentiles=(0, 1))
NICE = {"out_week_hours": "Outage hours this week", "out_prev_hours": "Outage hours last week", "out_cum_hours": "Cumulative outage hours",
        "gust_mean_ms": "Gust this week", "gust_cum": "Cumulative gust", "rain_mm": "Rain this week", "rain_cum": "Cumulative rain",
        "cold_ppt_mm": "Frozen precipitation", "tmin_c": "Lowest temperature", "rel_week": "Weeks since the storm", "log_pop": "Population",
        "log_density": "Population density", "coast_km": "Distance to the coast", "log_base_visits_pc": "Usual visits per person",
        "base_visitor_share": "Usual share of visitors from other counties", "base_out_share": "Usual share of visits made elsewhere",
        "net_in_hours": "Inbound exposure: visitors' home outage hours", "net_in_72": "Inbound exposure: visitors from 72+ h counties",
        "net_out_hours": "Outbound exposure: destinations' outage hours", "net_out_72": "Outbound exposure: destinations with 72+ h"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1.5, 1, 1]})
top = IMP.head(12).iloc[::-1]
is_net = [c not in F_LOCAL for c in top.index]
axes[0].barh([NICE[c] for c in top.index], top["mean"], color=[C_DARK if n else C_LIGHT for n in is_net], height=0.65)
axes[0].set_xlabel("Increase in mean absolute error when shuffled (log points)")
axes[0].set_title("Permutation importance (dark = network feature)")
axes[0].grid(axis="y", visible=False)
gv = lambda r: r["grid_values"][0] if "grid_values" in r else r["values"][0]
axes[1].plot(gv(pd_h), pd_h["average"][0], color=C_DARK, lw=2)
axes[1].axvline(HIT_HOURS, color=FAINT, lw=1, ls="--")
axes[1].set_xlabel("Outage hours this week")
axes[1].set_ylabel("Predicted change vs normal counties (log points)")
axes[1].set_title("Partial dependence: own outage")
axes[2].plot(np.asarray(gv(pd_e)) * 100, pd_e["average"][0], color=C_DARK, lw=2)
axes[2].set_xlabel("Share of usual visitors from counties with 72+ h of outage (%)")
axes[2].set_title("Partial dependence: inbound exposure")
fig.tight_layout()
fig.savefig(f"{OUT}/step4_fig3_what_it_learned.png"); plt.show()
tidy(IMP[["mean"]].rename(index=NICE).head(12), 3)

## F. Whose errors

Only county-weeks with 24+ hours of outage are used (leave-one-storm-out predictions, with Uri's transfer predictions reported separately). Residual = actual − predicted: below 0 means visits fell more than predicted, i.e. the model **underestimated the loss**. The 95% interval for the difference between rural and other counties comes from resampling counties.

In [ ]:
# 12. P4: residuals of rural and other counties
def gap(d, seed=SEED):
    d = d.assign(res=d["y"] - d["p_net"])
    by = d.groupby(["fips", "rural"])["res"].agg(["sum", "size"]).reset_index()
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(B_BOOT):
        s = by.iloc[rng.integers(0, len(by), len(by))]
        r_, o_ = s[s["rural"]], s[~s["rural"]]
        if r_["size"].sum() and o_["size"].sum():
            out.append(r_["sum"].sum() / r_["size"].sum() - o_["sum"].sum() / o_["size"].sum())
    point = d.loc[d["rural"], "res"].mean() - d.loc[~d["rural"], "res"].mean()
    return point, np.percentile(out, 2.5), np.percentile(out, 97.5), d

rows = []
for label, d in [("7 storms (held out)", EV_SET), ("Uri (transfer)", UE)]:
    a = d[(d["out_week_hours"] >= AFFECTED_HOURS) & d["rural"].notna()]
    if a["rural"].nunique() < 2:
        continue
    pt, lo, hi, a = gap(a)
    for grp_, sub in a.groupby("rural"):
        rows.append({"sample": label, "group": "rural (< 50k)" if grp_ else "other", "county_weeks": len(sub), "counties": sub["fips"].nunique(),
                     "mean_actual": sub["y"].mean(), "mean_predicted": sub["p_net"].mean(), "mean_residual": sub["res"].mean(),
                     "mae": sub["res"].abs().mean(), "gap_rural_minus_other": pt, "gap_lo": lo, "gap_hi": hi})
EQ = pd.DataFrame(rows)
EQ.to_csv(f"{OUT}/step4_equity.csv", index=False)
g7 = EQ[EQ["sample"] == "7 storms (held out)"].iloc[0] if len(EQ) else None
P4 = None if g7 is None else bool(g7["gap_hi"] < 0 or g7["gap_lo"] > 0)
P4_DIR = None if g7 is None else ("underestimates rural losses" if g7["gap_hi"] < 0 else ("overestimates rural losses" if g7["gap_lo"] > 0 else "no systematic difference"))
if g7 is not None:
    print(f"P4: rural − other mean residual {g7['gap_rural_minus_other']:+.1f} [{g7['gap_lo']:+.1f}, {g7['gap_hi']:+.1f}] → {P4_DIR}")
tidy(EQ.drop(columns=["gap_rural_minus_other", "gap_lo", "gap_hi"]).set_index(["sample", "group"]), 1)

In [ ]:
# 13. Checklist and conclusions
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

leak = TRANSFER in set(tr_all["event"]) or TRANSFER in set(LOSO["event"])
checks = [
    ("Evaluation county-weeks per held-out storm (minimum)", int(EV_SET.groupby("event").size().min()), "≥ 200",
     bool(EV_SET.groupby("event").size().min() >= 200)),
    ("Large drops in the pooled evaluation set", int(EV_SET["big"].sum()), "≥ 30", bool(EV_SET["big"].sum() >= 30)),
    ("Uri is in no training set", str(not leak), "True", not leak),
    ("At least 30 reference counties per event-week",
     int(D[D["keep"] & (D["group"] == "none")].groupby(["event", "rel_week"]).size().min()), "≥ 30",
     bool(D[D["keep"] & (D["group"] == "none")].groupby(["event", "rel_week"]).size().min() >= 30)),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step4_checklist.csv", index=False)
display(check_tab)

def yes_no(x):
    return "—" if x is None else ("yes" if x else "no")
p = pooled.loc["p_net"]
g7_txt = "—" if g7 is None else f"{g7['gap_rural_minus_other']:+.1f} [{g7['gap_lo']:+.1f}, {g7['gap_hi']:+.1f}]"
conclusions = pd.DataFrame([
    ("P1", f"R² {p['r2']:.3f}; Spearman {p['spearman']:.2f}; AP {p['ap']:.3f} (chance {p['base']:.3f})", yes_no(P1)),
    ("P2", f"AP difference {DIFF:+.3f} [{DIFF_LO:+.3f}, {DIFF_HI:+.3f}]; {WINS}/{ELIG}", yes_no(P2)),
    ("P3", f"R² {u['r2']:.3f}; Spearman {u['spearman']:.2f}; AP {u['ap']:.3f} (chance {u['base']:.3f})", yes_no(P3)),
    ("P4", g7_txt, P4_DIR if P4_DIR is not None else "—"),
], columns=["Question", "Value", "Conclusion"])
conclusions.to_csv(f"{OUT}/step4_conclusions.csv", index=False)
conclusions